In [14]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

## Breast Cancer Dataset

In [5]:

# ========================
print("\n=== Stabilité moyenne des explications (jeu de test) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


=== Stabilité moyenne des explications (jeu de test) ===
LinearRegression : 0.997 ± 0.003
DecisionTree     : 0.986 ± 0.103
NDT              : 0.991 ± 0.000


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper Decision Tree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Charger dataset WINE
# ========================
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ========================
# 2. Modèle global
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

# ========================
# 4. Stabilité d'une instance
# ========================
def explanation_stability(
    explainer,
    model_regressor,
    instance,
    n_repeats=5,
    num_features=None
):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            instance,
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0.0) for f in feature_names])
        explanations.append(vec)

    # Similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i + 1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num / denom if denom > 0 else 0.0)

    return np.mean(sims)

# ========================
# 5. Stabilité moyenne sur la dataset
# ========================
def average_stability_over_dataset(
    explainer,
    model_regressor,
    X_data,
    n_repeats=5,
    num_features=None
):
    scores = []

    for i in range(len(X_data)):
        score = explanation_stability(
            explainer=explainer,
            model_regressor=model_regressor,
            instance=X_data[i],
            n_repeats=n_repeats,
            num_features=num_features
        )
        scores.append(score)

    return np.mean(scores), np.std(scores)

# ========================
# 6. Calculs
# ========================
num_features = X_train.shape[1]

mean_lr, std_lr = average_stability_over_dataset(
    explainer_classic,
    LinearRegression(),
    X_test,
    n_repeats=5,
    num_features=num_features
)

mean_dt, std_dt = average_stability_over_dataset(
    explainer_ndt,
    DecisionTreeWrapper(),
    X_test,
    n_repeats=5,
    num_features=num_features
)

mean_ndt, std_ndt = average_stability_over_dataset(
    explainer_ndt,
    NDTRegressorWrapper(D=num_features, gammas=[1, 1]),
    X_test,
    n_repeats=5,
    num_features=num_features
)

# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (jeu de test) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


## Iris Dataset

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper Decision Tree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_iris()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ========================
# 2. Modèle global
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

# ========================
# 4. Stabilité d'une instance
# ========================
def explanation_stability(
    explainer,
    model_regressor,
    instance,
    n_repeats=5,
    num_features=4
):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            instance,
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0.0) for f in feature_names])
        explanations.append(vec)

    # Similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i + 1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num / denom if denom > 0 else 0.0)

    return np.mean(sims)

# ========================
# 5. Stabilité moyenne sur la dataset
# ========================
def average_stability_over_dataset(
    explainer,
    model_regressor,
    X_data,
    n_repeats=5,
    num_features=4
):
    scores = []

    for i in range(len(X_data)):
        score = explanation_stability(
            explainer=explainer,
            model_regressor=model_regressor,
            instance=X_data[i],
            n_repeats=n_repeats,
            num_features=num_features
        )
        scores.append(score)

    return np.mean(scores), np.std(scores)

# ========================
# 6. Calculs
# ========================
mean_lr, std_lr = average_stability_over_dataset(
    explainer_classic,
    LinearRegression(),
    X_test,
    n_repeats=5,
    num_features=4
)

mean_dt, std_dt = average_stability_over_dataset(
    explainer_ndt,
    DecisionTreeWrapper(),
    X_test,
    n_repeats=5,
    num_features=4
)

mean_ndt, std_ndt = average_stability_over_dataset(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1, 1]),
    X_test,
    n_repeats=5,
    num_features=X_train.shape[1]
)

# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (jeu de test) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


## Wine Dataset

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper Decision Tree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Charger dataset WINE
# ========================
data = load_wine()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ========================
# 2. Modèle global
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

# ========================
# 4. Stabilité d'une instance
# ========================
def explanation_stability(
    explainer,
    model_regressor,
    instance,
    n_repeats=5,
    num_features=None
):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            instance,
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0.0) for f in feature_names])
        explanations.append(vec)

    # Similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i + 1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num / denom if denom > 0 else 0.0)

    return np.mean(sims)

# ========================
# 5. Stabilité moyenne sur la dataset
# ========================
def average_stability_over_dataset(
    explainer,
    model_regressor,
    X_data,
    n_repeats=5,
    num_features=None
):
    scores = []

    for i in range(len(X_data)):
        score = explanation_stability(
            explainer=explainer,
            model_regressor=model_regressor,
            instance=X_data[i],
            n_repeats=n_repeats,
            num_features=num_features
        )
        scores.append(score)

    return np.mean(scores), np.std(scores)

# ========================
# 6. Calculs
# ========================
num_features = X_train.shape[1]

mean_lr, std_lr = average_stability_over_dataset(
    explainer_classic,
    LinearRegression(),
    X_test,
    n_repeats=5,
    num_features=num_features
)

mean_dt, std_dt = average_stability_over_dataset(
    explainer_ndt,
    DecisionTreeWrapper(),
    X_test,
    n_repeats=5,
    num_features=num_features
)

mean_ndt, std_ndt = average_stability_over_dataset(
    explainer_ndt,
    NDTRegressorWrapper(D=num_features, gammas=[1, 1]),
    X_test,
    n_repeats=5,
    num_features=num_features
)

# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (jeu de test) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


## California Housing Dataset

In [20]:
# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (Diabetes – 200 points) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")



=== Stabilité moyenne des explications (Diabetes – 200 points) ===
LinearRegression : 0.999 ± 0.000
DecisionTree     : 0.999 ± 0.000
NDT              : 0.973 ± 0.001


In [19]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper Decision Tree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Charger dataset DIABETES
# ========================
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

# ========================
# 🔹 Sélection de 200 points du jeu de test
# ========================
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=100, replace=False)
X_test_200 = X_test[idx]

# ========================
# 2. Modèle global
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="regression"
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="regression"
)

# ========================
# 4. Stabilité d'une instance
# ========================
def explanation_stability(
    explainer,
    model_regressor,
    instance,
    n_repeats=5,
    num_features=None
):
    explanations = []

    if num_features is None:
        num_features = X_train.shape[1]

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            instance,
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0.0) for f in feature_names])
        explanations.append(vec)

    sims = []
    for i in range(len(explanations)):
        for j in range(i + 1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num / denom if denom > 0 else 0.0)

    return np.mean(sims)

# ========================
# 5. Stabilité moyenne sur la dataset
# ========================
def average_stability_over_dataset(
    explainer,
    model_regressor,
    X_data,
    n_repeats=5,
    num_features=None
):
    scores = []

    for i in range(len(X_data)):
        score = explanation_stability(
            explainer=explainer,
            model_regressor=model_regressor,
            instance=X_data[i],
            n_repeats=n_repeats,
            num_features=num_features
        )
        scores.append(score)

    return np.mean(scores), np.std(scores)

# ========================
# 6. Calculs (SUR 200 POINTS)
# ========================
num_features = X_train.shape[1]

mean_lr, std_lr = average_stability_over_dataset(
    explainer_classic,
    LinearRegression(),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

mean_dt, std_dt = average_stability_over_dataset(
    explainer_ndt,
    DecisionTreeWrapper(),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

mean_ndt, std_ndt = average_stability_over_dataset(
    explainer_ndt,
    NDTRegressorWrapper(D=num_features, gammas=[1, 1]),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (Diabetes – 200 points) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 959us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 930us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 944us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

=== Stabilité moyenne des explications (Diabetes – 200 points) ===
LinearRegression : 0.999 ± 0.000
DecisionTree     : 0.999 ± 0.000
NDT              : 0.973 ± 0.001


In [5]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=8):
    explanations = []
    if num_features is None:
        num_features = X_train.shape[1]
    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[1,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")

mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 917us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 915us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 860us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 960us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.999
DecisionTree: 0.999
NDT: 0.974


## Diabetes Dataset

In [10]:
# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (Diabetes – 200 points) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


=== Stabilité moyenne des explications (Diabetes – 200 points) ===
LinearRegression : 0.999 ± 0.000
DecisionTree     : 0.999 ± 0.000
NDT              : 0.998 ± 0.000


In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper Decision Tree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Charger dataset DIABETES
# ========================
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

# ========================
# 🔹 Sélection de 200 points du jeu de test
# ========================
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=100, replace=False)
X_test_200 = X_test[idx]

# ========================
# 2. Modèle global
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="regression"
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="regression"
)

# ========================
# 4. Stabilité d'une instance
# ========================
def explanation_stability(
    explainer,
    model_regressor,
    instance,
    n_repeats=5,
    num_features=None
):
    explanations = []

    if num_features is None:
        num_features = X_train.shape[1]

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            instance,
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0.0) for f in feature_names])
        explanations.append(vec)

    sims = []
    for i in range(len(explanations)):
        for j in range(i + 1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num / denom if denom > 0 else 0.0)

    return np.mean(sims)

# ========================
# 5. Stabilité moyenne sur la dataset
# ========================
def average_stability_over_dataset(
    explainer,
    model_regressor,
    X_data,
    n_repeats=5,
    num_features=None
):
    scores = []

    for i in range(len(X_data)):
        score = explanation_stability(
            explainer=explainer,
            model_regressor=model_regressor,
            instance=X_data[i],
            n_repeats=n_repeats,
            num_features=num_features
        )
        scores.append(score)

    return np.mean(scores), np.std(scores)

# ========================
# 6. Calculs (SUR 200 POINTS)
# ========================
num_features = X_train.shape[1]

mean_lr, std_lr = average_stability_over_dataset(
    explainer_classic,
    LinearRegression(),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

mean_dt, std_dt = average_stability_over_dataset(
    explainer_ndt,
    DecisionTreeWrapper(),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

mean_ndt, std_ndt = average_stability_over_dataset(
    explainer_ndt,
    NDTRegressorWrapper(D=num_features, gammas=[1, 1]),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (Diabetes – 200 points) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


## Digits Dataset

In [7]:
# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (200 points du jeu de test) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


=== Stabilité moyenne des explications (200 points du jeu de test) ===
LinearRegression : 0.983 ± 0.005
DecisionTree     : 0.983 ± 0.108
NDT              : 0.931 ± 0.007


In [ ]:
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper Decision Tree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Charger dataset DIGITS
# ========================
data = fetch_covtype()
X = data.data[:2000]
y = data.target[:2000]
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ========================
# 🔹 Sélection de 200 points du jeu de test
# ========================
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=100, replace=False)
X_test_200 = X_test[idx]

# ========================
# 2. Modèle global
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode="classification"
)

# ========================
# 4. Stabilité d'une instance
# ========================
def explanation_stability(
    explainer,
    model_regressor,
    instance,
    n_repeats=5,
    num_features=None
):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            instance,
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0.0) for f in feature_names])
        explanations.append(vec)

    sims = []
    for i in range(len(explanations)):
        for j in range(i + 1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num / denom if denom > 0 else 0.0)

    return np.mean(sims)

# ========================
# 5. Stabilité moyenne sur la dataset
# ========================
def average_stability_over_dataset(
    explainer,
    model_regressor,
    X_data,
    n_repeats=5,
    num_features=None
):
    scores = []

    for i in range(len(X_data)):
        score = explanation_stability(
            explainer=explainer,
            model_regressor=model_regressor,
            instance=X_data[i],
            n_repeats=n_repeats,
            num_features=num_features
        )
        scores.append(score)

    return np.mean(scores), np.std(scores)

# ========================
# 6. Calculs (SUR 200 POINTS)
# ========================
num_features = X_train.shape[1]

mean_lr, std_lr = average_stability_over_dataset(
    explainer_classic,
    LinearRegression(),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

mean_dt, std_dt = average_stability_over_dataset(
    explainer_ndt,
    DecisionTreeWrapper(),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

mean_ndt, std_ndt = average_stability_over_dataset(
    explainer_ndt,
    NDTRegressorWrapper(D=num_features, gammas=[1, 1]),
    X_test_200,
    n_repeats=5,
    num_features=num_features
)

# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (200 points du jeu de test) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


## Covtype Dataset

In [5]:
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeNdtExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = fetch_covtype()
X = data.data[:2000]
y = data.target[:2000]
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=13):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], gammas=[100,1]),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.990
DecisionTree: 0.997
NDT: 0.956


## Ames Housing Dataset

In [17]:
# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (House Prices – 100 points) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


=== Stabilité moyenne des explications (House Prices – 100 points) ===
LinearRegression : 0.998 ± 0.000
DecisionTree     : 0.996 ± 0.041
NDT              : 0.990 ± 0.013


In [16]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# ========================
# LIME
# ========================
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeNdtExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper Decision Tree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# 1. Charger dataset HOUSE PRICES
# ========================
data = fetch_openml(name="house_prices", as_frame=True)

X = data.data.select_dtypes(include=[np.number]).dropna(axis=1)
y = data.target.astype(float)

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

# ========================
# 🔹 Sélection de 100 points du jeu de test
# ========================
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=100, replace=False)

X_test_100 = X_test.iloc[idx]

# ========================
# 2. Modèle global
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train.values,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="regression"
)

explainer_ndt = LimeNdtExplainer(
    X_train.values,
    feature_names=feature_names,
    discretize_continuous=False,
    mode="regression"
)

# ========================
# 4. Stabilité d'une instance
# ========================
def explanation_stability(
    explainer,
    model_regressor,
    instance,
    n_repeats=5,
    num_features=None
):
    explanations = []

    if num_features is None:
        num_features = X_train.shape[1]

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            instance,
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0.0) for f in feature_names])
        explanations.append(vec)

    sims = []
    for i in range(len(explanations)):
        for j in range(i + 1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num / denom if denom > 0 else 0.0)

    return np.mean(sims)

# ========================
# 5. Stabilité moyenne sur la dataset (100 instances)
# ========================
def average_stability_over_dataset(
    explainer,
    model_regressor,
    X_data,
    n_repeats=5,
    num_features=None
):
    scores = []

    for i in range(len(X_data)):
        instance = X_data.iloc[i].values  # ✅ Pandas → NumPy
        score = explanation_stability(
            explainer=explainer,
            model_regressor=model_regressor,
            instance=instance,
            n_repeats=n_repeats,
            num_features=num_features
        )
        scores.append(score)

    return np.mean(scores), np.std(scores)

# ========================
# 6. Calculs
# ========================
num_features = X_train.shape[1]

mean_lr, std_lr = average_stability_over_dataset(
    explainer_classic,
    LinearRegression(),
    X_test_100,
    n_repeats=5,
    num_features=num_features
)

mean_dt, std_dt = average_stability_over_dataset(
    explainer_ndt,
    DecisionTreeWrapper(),
    X_test_100,
    n_repeats=5,
    num_features=num_features
)

mean_ndt, std_ndt = average_stability_over_dataset(
    explainer_ndt,
    NDTRegressorWrapper(D=num_features, gammas=[1, 1]),
    X_test_100,
    n_repeats=5,
    num_features=num_features
)

# ========================
# 7. Résultats
# ========================
print("\n=== Stabilité moyenne des explications (House Prices – 100 points) ===")
print(f"LinearRegression : {mean_lr:.3f} ± {std_lr:.3f}")
print(f"DecisionTree     : {mean_dt:.3f} ± {std_dt:.3f}")
print(f"NDT              : {mean_ndt:.3f} ± {std_ndt:.3f}")


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomFor

mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 914us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 959us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step

=== Stabilité moyenne des explications (House Prices – 100 points) ===
LinearRegression : 0.998 ± 0.000
DecisionTree     : 0.996 ± 0.041
NDT              : 0.990 ± 0.013
